In [ ]:
# This notebook is generic and driven by a configuration file.
# The name of the configuration to use is passed in via a widget.

# 1. Create a widget to accept the configuration name.
dbutils.widgets.text("config_name", "", "Name of the config file in /config (e.g., ewm_ddl_ext)")

# 2. Retrieve the value from the widget.
config_name = dbutils.widgets.get("config_name")

if not config_name:
    raise ValueError("Widget 'config_name' must not be empty. Please provide a config file name.")

print(f"✅ Using configuration for: '{config_name}'")

In [ ]:
# This cell reads the shared configuration file and makes it available to the rest of the notebook.
import yaml

# --- CONFIGURATION ---
# The path is dynamically constructed based on the widget value.
config_path = f"../config/{config_name}.yml"

try:
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
except FileNotFoundError:
    raise FileNotFoundError(f"Config file not found at path: {config_path}. Make sure the widget name matches the file name.")

# Extract specific sections for clarity
validation_config = config['job_a_validation']
shared_config = config

print("✅ Configuration loaded successfully.")

In [ ]:
# ===================================================================================
#
# JOB A - POST-ACTIVATION VALIDATION SCRIPT (AUTOMATED)
#
# ===================================================================================

from pyspark.sql import functions as F

# ===================================================================================
# --- CONFIGURATION (from config file) ---
# ===================================================================================

target_table_name_full = validation_config["target_table_name_full"]
control_table_schema = shared_config["control_table_schema"]

# ===================================================================================
# --- SCRIPT EXECUTION STARTS HERE ---
# ===================================================================================

### Section 1: Initialization ###

# --- Table Names ---
schema_transition_table = f"{control_table_schema}.schema_transition"
schema_store_columns_table = f"{control_table_schema}.schema_store_columns"

# Use the simple table name (the part after the last dot) for all metadata lookups.
simple_table_name = target_table_name_full.split('.')[-1]

print(f"✅ Starting Post-Activation Validation for:")
print(f"   - Target Table: {target_table_name_full}")
print(f"   - Simple Name Key: {simple_table_name}")

failures = []

### Section 2: Validation Checks ###

# --- AUTOMATICALLY DETECT ACTIVE HASH ---
print("\n🔎 Finding the current 'ACTIVE' schema hash...")
try:
    active_hash_row = spark.table(schema_transition_table).filter(f"target_table_name = '{simple_table_name}' AND event_type = 'ACTIVE'").first()
    if not active_hash_row:
        raise RuntimeError(f"CRITICAL: Could not find an 'ACTIVE' schema for table '{simple_table_name}'. Cannot proceed with validation.")
    
    newly_activated_hash = active_hash_row['schema_hash']
    print(f"   - Hash to Validate: {newly_activated_hash}")

except Exception as e:
    failures.append(f"CRITICAL FAILURE during active hash detection: {e}")
    newly_activated_hash = None # Ensure validation fails if we can't get the hash

if newly_activated_hash:
    # --- Check 1: State Coherency in 'schema_transition' table ---
    print(f"\n🔄 Running Check 1: State Coherency in 'schema_transition' (for key: '{simple_table_name}')...")
    try:
        transition_df = spark.table(schema_transition_table).filter(f"target_table_name = '{simple_table_name}'")
        active_rows = transition_df.filter("event_type = 'ACTIVE'").collect()

        if len(active_rows) != 1:
            failures.append(f"FAILURE: Expected exactly 1 'ACTIVE' schema, but found {len(active_rows)}.")
        else:
            if active_rows[0]["schema_hash"] != newly_activated_hash:
                failures.append(f"FAILURE: The 'ACTIVE' schema hash is '{active_rows[0]['schema_hash']}', but expected '{newly_activated_hash}'.")
            else:
                print("  - PASSED: Exactly one schema is 'ACTIVE' and its hash is correct.")

        non_archived_rows = transition_df.filter("event_type NOT IN ('ACTIVE', 'ARCHIVED')").count()
        if non_archived_rows > 0:
            failures.append(f"FAILURE: Found {non_archived_rows} rows that were not 'ACTIVE' or 'ARCHIVED'. All old schemas must be archived.")
        else:
            print("  - PASSED: All non-active schemas are correctly 'ARCHIVED'.")

    except Exception as e:
        failures.append(f"CRITICAL FAILURE during State Coherency check: {e}")

    # --- Check 2: The "Backfill Completeness" Check ---
    print(f"\n🔄 Running Check 2: Backfill Completeness in 'schema_transition' (for key: '{simple_table_name}')...")
    try:
        backfill_status_row = spark.table(schema_transition_table).filter(f"target_table_name = '{simple_table_name}' AND schema_hash = '{newly_activated_hash}'").first()
        if not backfill_status_row:
            failures.append(f"FAILURE: Cannot find the transaction row for the new hash '{newly_activated_hash}'.")
        else:
            status = backfill_status_row["backfill_status"]
            if status not in ["COMPLETE", "NOT_APPLICABLE"]:
                failures.append(f"FAILURE: Expected backfill status to be 'COMPLETE' or 'NOT_APPLICABLE', but found '{status}'.")
            else:
                print(f"  - PASSED: Backfill status is correctly set to '{status}'.")
    except Exception as e:
        failures.append(f"CRITICAL FAILURE during Backfill Completeness check: {e}")

    # --- Check 3: The "Blueprint Existence" Check ---
    print(f"\n🔄 Running Check 3: Blueprint Existence in 'schema_store_columns' (for key: '{simple_table_name}')...")
    try:
        blueprint_count = spark.table(schema_store_columns_table).filter(f"target_table_name = '{simple_table_name}' AND contract_hash = '{newly_activated_hash}'").count()
        if blueprint_count == 0:
            failures.append(f"FAILURE: No column blueprint found in '{schema_store_columns_table}' for the new hash '{newly_activated_hash}'. Job B will fail.")
        else:
            print(f"  - PASSED: Found {blueprint_count} blueprint columns for the new hash.")
    except Exception as e:
        failures.append(f"CRITICAL FAILURE during Blueprint Existence check: {e}")


### Section 3: Final Verdict ###

if failures:
    print("\n\n❌ VALIDATION FAILED. The system may be in an inconsistent state.")
    for f in failures:
        print(f"  - {f}")
    raise Exception("Post-activation validation checks failed. Please review the errors.")
else:
    success_msg = "✅ All post-activation validation checks passed successfully."
    print(f"\n\n{success_msg}")